In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gymnasium as gym

In [2]:
SEED = 0

np.random.seed(SEED)
torch.manual_seed(SEED)

env = gym.make(
    "FrozenLake-v1",
    map_name="4x4",
    is_slippery=False
)

N_STATES = 16
N_ACTIONS = 4

TERMINAL = {5, 7, 11, 12, 15}

In [3]:
def one_hot(s):
    v = np.zeros(N_STATES, dtype=np.float32)
    v[s] = 1.0
    return v

In [4]:
def behavior_policy():
    if np.random.rand() < 0.3:
        return env.action_space.sample()
    
    return np.random.choice([1, 2])

In [5]:
S, A, R, S2, D = [], [], [], [], []

for _ in range(300):
    s, _ = env.reset()
    done = False

    while not done:
        a = behavior_policy()

        s2, r, term, trunc, _ = env.step(a)
        done = term or trunc

        S.append(s)
        A.append(a)
        R.append(r)
        S2.append(s2)
        D.append(float(term))

        s = s2

S = np.array(S)
A = np.array(A)
S2 = np.array(S2)

R = np.array(R, dtype=np.float32)
D = np.array(D, dtype=np.float32)

print("Dataset collected successfully!")
print("Total transitions:", len(S))

Dataset collected successfully!
Total transitions: 1219


In [6]:
pair_counts = np.zeros((N_STATES, N_ACTIONS), dtype=int)

for s, a in zip(S, A):
    pair_counts[s, a] += 1

valid = [
    s for s in range(N_STATES)
    if s not in TERMINAL
]

common = [
    (s, a)
    for s in valid
    for a in range(N_ACTIONS)
    if pair_counts[s, a] >= 5
]

rare = [
    (s, a)
    for s in valid
    for a in range(N_ACTIONS)
    if pair_counts[s, a] < 5
]

print("Common pairs:", len(common))
print("Rare/unseen pairs:", len(rare))

Common pairs: 35
Rare/unseen pairs: 9


In [7]:
class QNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(N_STATES, 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, N_ACTIONS)
        )

    def forward(self, x):
        return self.net(x)

In [8]:
def cql_loss(
    q_net,
    target_net,
    s,
    a,
    r,
    s2,
    done,
    gamma,
    alpha
):

    q_all = q_net(s)

    q_data = q_all.gather(
        1,
        a.unsqueeze(1)
    ).squeeze(1)

    with torch.no_grad():

        target = (
            r
            + gamma * (1 - done)
            * target_net(s2).max(dim=1).values
        )

    bellman = F.mse_loss(
        q_data,
        target
    )

    cql_pen = (
        torch.logsumexp(q_all, dim=1)
        - q_data
    ).mean()

    return bellman + alpha * cql_pen

In [9]:
S_T = torch.tensor(
    np.array([one_hot(x) for x in S])
)

S2_T = torch.tensor(
    np.array([one_hot(x) for x in S2])
)

A_T = torch.tensor(
    A,
    dtype=torch.long
)

R_T = torch.tensor(R)

D_T = torch.tensor(D)

print("Dataset converted to tensors successfully!")

Dataset converted to tensors successfully!


In [10]:
def train(
    alpha,
    steps=3000,
    batch=64,
    lr=1e-3,
    gamma=0.99
):

    torch.manual_seed(SEED)
    np.random.seed(SEED)

    q = QNet()
    tgt = QNet()

    tgt.load_state_dict(
        q.state_dict()
    )

    opt = torch.optim.Adam(
        q.parameters(),
        lr=lr
    )

    for step in range(steps):

        i = np.random.randint(
            0,
            len(S),
            batch
        )

        loss = cql_loss(
            q,
            tgt,
            S_T[i],
            A_T[i],
            R_T[i],
            S2_T[i],
            D_T[i],
            gamma,
            alpha
        )

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % 100 == 0:
            tgt.load_state_dict(
                q.state_dict()
            )

    return q

In [11]:
def q_table(q):

    with torch.no_grad():

        states = torch.tensor(
            np.array([
                one_hot(s)
                for s in range(N_STATES)
            ])
        )

        return q(states).numpy()

In [12]:
def evaluate(q):

    s, _ = env.reset()

    total = 0.0
    done = False
    steps = 0

    while not done and steps < 50:

        with torch.no_grad():

            state_tensor = torch.tensor(
                one_hot(s)
            ).unsqueeze(0)

            a = q(
                state_tensor
            ).argmax().item()

        s, r, term, trunc, _ = env.step(a)

        done = term or trunc

        total += r
        steps += 1

    return total

In [13]:
def avg_q(qt, pairs):

    return float(
        np.mean([
            qt[s, a]
            for s, a in pairs
        ])
    )

In [14]:
print("Training Offline DQN...")

dqn_model = train(alpha=0.0)

dqn_q = q_table(dqn_model)

dqn_ret = evaluate(dqn_model)

print("Offline DQN training completed!")

Training Offline DQN...
Offline DQN training completed!


In [15]:
print("Training CQL...")

cql_model = train(alpha=1.0)

cql_q = q_table(cql_model)

cql_ret = evaluate(cql_model)

print("CQL training completed!")

Training CQL...
CQL training completed!


In [16]:
print(
    f"Dataset: {len(S)} transitions"
)

print(
    f"Left used: {100*np.mean(A == 0):.0f}%"
)

print(
    f"Down used: {100*np.mean(A == 1):.0f}%"
)

print(
    f"Right used: {100*np.mean(A == 2):.0f}%"
)

print(
    f"Up used: {100*np.mean(A == 3):.0f}%"
)

print(
    f"\nRare/unseen (state, action) pairs: "
    f"{len(rare)} of {len(rare) + len(common)}"
)

Dataset: 1219 transitions
Left used: 7%
Down used: 44%
Right used: 42%
Up used: 7%

Rare/unseen (state, action) pairs: 9 of 44


In [17]:
print(
    f"{'':<15}"
    f"{'Return':>10}"
    f"{'Avg Q common':>18}"
    f"{'Avg Q rare':>15}"
    f"{'Rare Q > 1.0':>18}"
)

for name, qt, ret in [
    ("Offline DQN", dqn_q, dqn_ret),
    ("CQL", cql_q, cql_ret)
]:

    over = sum(
        qt[s, a] > 1.0
        for s, a in rare
    )

    print(
        f"{name:<15}"
        f"{ret:>10.1f}"
        f"{avg_q(qt, common):>18.3f}"
        f"{avg_q(qt, rare):>15.3f}"
        f"{over:>10d}/{len(rare)}"
    )

print(
    "\nNote: The maximum possible reward is 1.0, "
    "so Q-values above 1.0 indicate overestimation."
)

                   Return      Avg Q common     Avg Q rare      Rare Q > 1.0
Offline DQN           1.0             0.773          0.752         0/9
CQL                   0.0             1.012          0.559         1/9

Note: The maximum possible reward is 1.0, so Q-values above 1.0 indicate overestimation.


In [18]:
print("========== FINAL RESULT ==========")

print(f"Dataset transitions : {len(S)}")
print(f"Rare/unseen pairs   : {len(rare)}")

print("\nOffline DQN")
print(f"Return              : {dqn_ret:.1f}")
print(f"Average Q (common)  : {avg_q(dqn_q, common):.3f}")
print(f"Average Q (rare)    : {avg_q(dqn_q, rare):.3f}")

dqn_over = sum(
    dqn_q[s, a] > 1.0
    for s, a in rare
)

print(
    f"Rare Q > 1.0        : "
    f"{dqn_over}/{len(rare)}"
)

print("\nCQL")
print(f"Return              : {cql_ret:.1f}")
print(f"Average Q (common)  : {avg_q(cql_q, common):.3f}")
print(f"Average Q (rare)    : {avg_q(cql_q, rare):.3f}")

cql_over = sum(
    cql_q[s, a] > 1.0
    for s, a in rare
)

print(
    f"Rare Q > 1.0        : "
    f"{cql_over}/{len(rare)}"
)

========== FINAL RESULT ==========
Dataset transitions : 1219
Rare/unseen pairs   : 9

Offline DQN
Return              : 1.0
Average Q (common)  : 0.773
Average Q (rare)    : 0.752
Rare Q > 1.0        : 0/9

CQL
Return              : 0.0
Average Q (common)  : 1.012
Average Q (rare)    : 0.559
Rare Q > 1.0        : 1/9
